# CSA — mở rộng nhãn TRAIN từ 10.000 lên 30.000 ảnh (CPU)
**Accelerator None**, Internet ON. Add Input:
- `ducanh2403/csa-full-roi-karpathy-v134`
- `vuthetam/mscoco-2014`

Giữ nguyên quy tắc nhãn/IoU/box và exact10k cũ làm prefix của30k.
Không chạy detector/CLIP/mô hình; không dùng VALID/TEST. Tải annotation chính thức,
kiểm hash trước khi đọc. Không cần checkpoint, API secret hoặc cache prompt.

Đây chỉ là tạo nhãn: chưa chạy hai nhánh GPU. Đã chốt sampler lần lượt1caption/ảnh,
nên cả30k ảnh đều được dùng trong64.000 lượt học ở bướcGPU sau.
Không hứa tăng CIDEr từ việc tăng pool. Old10k và hai pilot được giữ nguyên.

Save & Run All, gửi `csa_grounding_data_audit_30k_v2_review.zip`.
Giữ TOÀN BỘ folder output, đặc biệt mentions.jsonl và train_instances.json,
để lưu Dataset cho hai nhánh sau. Không tạo lại fullROI hay Dataset150.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile
from pathlib import Path
COMMIT='24ccd81a411ee0e05cd1abd61fbc8b1044b56075'
HASHES={'grounding_labels.py': '9949320f8aa29220dd5d825ce6ee67a0a87b1a82422639ce5457ee736c2db86c', 'run_grounding_audit.py': 'e1164bf171914e5e5ff516af9349fadaad3afcefabdf16f1b62450dd989a69fa', 'test_grounding_labels.py': '11ceed8178718552bd5e50a828f0fa14ed15a899b2e72f7a058093b0cad828f5', 'SAVE_FULL_ROI_V134_PACKAGE.json': 'a73636ca75806d6b1d7b4d88f80f3ce1c59b3061eb3e9a5e730097ddd89c505c', 'GROUNDED_ATTENTION_AUDIT_PROTOCOL.md': 'f8570d9585df465c95f7d3b2ae1570e70606207e0a42058860722e2e6100d5f9', 'GROUNDING_30K_PROTOCOL.md': 'd376c3922aed18da014faf422341aa0574cf8588f336442a6d06693cba160213'}
FULL_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-roi-karpathy-v134')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
PREVIOUS_SOURCE=globals().get('SOURCE')
ANNOTATION_ARCHIVE=(Path(PREVIOUS_SOURCE)/'work'/'annotations_trainval2014.zip') if PREVIOUS_SOURCE else None
OUTPUT=Path('/kaggle/working/csa_grounding_data_audit_30k_v2')
if OUTPUT.exists():raise RuntimeError('Preserve previous output; use a fresh session.')
if not FULL_ROOT.is_dir() or not COCO_JSON.is_file():raise FileNotFoundError('Attach both datasets; check FULL_ROOT and COCO_JSON before running.')
OUTPUT.mkdir(parents=True)
SOURCE=Path(tempfile.mkdtemp(prefix='csa-grounding-audit-'))
LOG=OUTPUT/'notebook.log'
CONFIG=SOURCE/'config.json'
CONFIG.write_text(json.dumps(dict(full_root=str(FULL_ROOT),dataset_json=str(COCO_JSON),output=str(OUTPUT),work=str(SOURCE/'work'),image_count=30000,annotation_archive=str(ANNOTATION_ARCHIVE) if ANNOTATION_ARCHIVE else None)),encoding='utf-8')
for name,expected in HASHES.items():
    data=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{name}',timeout=90).read()
    if hashlib.sha256(data).hexdigest()!=expected:raise ValueError('Changed source '+name)
    (SOURCE/name).write_bytes(data)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT,sha256=HASHES),indent=2),encoding='utf-8')
print('CPU audit source ready; no GPU/checkpoint/API token needed.',flush=True)


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('CPU audit active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    run([sys.executable,'-m','pip','install','-q','ijson==3.4.0'],timeout=300)
    run([sys.executable,'-u',SOURCE/'test_grounding_labels.py'],timeout=120)
    run([sys.executable,'-u',SOURCE/'run_grounding_audit.py','--config',CONFIG],timeout=7200)
    result=json.loads((OUTPUT/'summary.json').read_text(encoding='utf-8'))
    complete=result['status']=='CSA_GROUNDING_DATA_AUDIT_COMPLETE'
    if not complete:raise RuntimeError('Audit incomplete')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8')
    raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,model_executed=False,training_ready=False,manual_review_pending=True)),encoding='utf-8')
    # Keep full labels/filtered annotations in Output; send only review/sample metadata.
    archive=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.iterdir()):
            if p.is_file() and p.name not in ('mentions.jsonl','train10k_instances.json','train_instances.json'):z.write(p,OUTPUT.name+'/'+p.name)
    print('SEND REVIEW:',archive,flush=True)
    print('KEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    print('30k labels complete; send review ZIP before preparing matched GPU arms.' if complete else 'INCOMPLETE: send ZIP/log.',flush=True)
